In [1]:
!pip install -q datasets soundfile transformers accelerate torch resemblyzer librosa pandas scipy parler_tts

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.


In [2]:
!pip install protobuf==4.25.9

In [2]:
!pip install --upgrade sympy

In [3]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import os
from pathlib import Path
from scipy.signal import resample
from resemblyzer import VoiceEncoder, preprocess_wav
import glob
import zipfile
from google.colab import drive

# ================== NEW: Indic-Parler-TTS Imports & Config ==================
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

# Config (unchanged)
NUM_SAMPLES_PER_EMOTION = 10
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Indic-Parler-TTS model to {device}... (this may take 1-2 minutes first time)")

# Load Indic-Parler-TTS (male Urdu voice using named speaker "Rohit" for perfect consistency)
model = ParlerTTSForConditionalGeneration.from_pretrained("ai4bharat/indic-parler-tts").to(device)
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts")
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)

encoder = VoiceEncoder()

# Fixed male Urdu description using "Rohit" (best consistency + naturalness)
MALE_URDU_DESCRIPTION = (
    "Rohit's voice is clear and natural with a moderate speed and pitch. "
    "The recording is of very high quality, with the speaker's voice sounding very close up "
    "and no background noise."
)

print("✅ Indic-Parler-TTS loaded successfully (male Urdu voice: Rohit)")

def run_evaluation(dataset_name, samples_list, text_key):
    base_dir = f"mushra_{dataset_name}"
    for sub in ["reference", "generated", "anchor"]:
        os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

    results = []
    # Increased limit to 60 to accommodate 10 samples per 6 emotions
    for i in range(min(len(samples_list), 60)):
        try:
            sample = samples_list[i]
            text = sample[text_key]
            ref_array = np.array(sample['audio']['array']).astype(np.float32)
            ref_sr = sample['audio']['sampling_rate']

            if ref_array.ndim > 1:
                ref_array = ref_array.mean(axis=1)

            ref_path = f"{base_dir}/reference/sample_{i}.wav"
            gen_path = f"{base_dir}/generated/sample_{i}.wav"
            anc_path = f"{base_dir}/anchor/sample_{i}.wav"

            # Save Reference (unchanged)
            sf.write(ref_path, ref_array, ref_sr)

            # ================== NEW: Indic-Parler-TTS Generation ==================
            description_input_ids = description_tokenizer(
                MALE_URDU_DESCRIPTION, return_tensors="pt"
            ).to(device)
            prompt_input_ids = tokenizer(text, return_tensors="pt").to(device)

            with torch.no_grad():
                generation = model.generate(
                    input_ids=description_input_ids.input_ids,
                    attention_mask=description_input_ids.attention_mask,
                    prompt_input_ids=prompt_input_ids.input_ids,
                    prompt_attention_mask=prompt_input_ids.attention_mask,
                    do_sample=True,          # better naturalness
                    temperature=0.8
                )

            gen_speech = generation.cpu().numpy().squeeze()
            gen_sr = model.config.sampling_rate
            sf.write(gen_path, gen_speech, gen_sr)

            # Create Anchor (8kHz cycle) - unchanged
            target_len_8k = int(len(ref_array) * 8000 / ref_sr)
            downsampled = resample(ref_array, target_len_8k)
            upsampled = resample(downsampled, len(ref_array))
            sf.write(anc_path, upsampled, ref_sr)

            # Similarity (unchanged)
            wav_ref = preprocess_wav(Path(ref_path))
            wav_gen = preprocess_wav(Path(gen_path))
            emb_ref = encoder.embed_utterance(wav_ref)
            emb_gen = encoder.embed_utterance(wav_gen)
            sim = np.dot(emb_ref, emb_gen) / (np.linalg.norm(emb_ref) * np.linalg.norm(emb_gen))

            results.append({
                "sample_id": i,
                "emotion": sample.get("emotion", "unknown"),
                "urdu_text": text,
                "ref_gen_similarity": float(sim)
            })
            print(f"Sample {i+1} | Emotion: {sample.get('emotion','?')} | Similarity={sim:.4f}")

        except Exception as e:
            print(f"Error in sample {i}: {e}")
            continue

    return results

Loading Indic-Parler-TTS model to cuda... (this may take 1-2 minutes first time)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

generation_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loaded the voice encoder model on cuda in 0.19 seconds.
✅ Indic-Parler-TTS loaded successfully (male Urdu voice: Rohit)


In [5]:
import zipfile
import os
from google.colab import drive

# Mount Google Drive to access the zip file
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/urduser.zip'
extract_to = '/content/UrduSER/'

if os.path.exists(zip_path):
    print(f"Unzipping {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("✅ Unzipped successfully!")
    # List contents to verify structure
    print("Folder contents:", os.listdir(extract_to))
else:
    print(f"❌ Could not find {zip_path} in your Drive root.")
    print("Please ensure the file is named 'urduser.zip' and is located in the top-level 'My Drive' folder.")

Mounted at /content/drive
Unzipping /content/drive/MyDrive/urduser.zip...
✅ Unzipped successfully!
Folder contents: ['UrduSER A Dataset for Urdu Speech Emotion Recognition']


In [6]:
import os

def list_all_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')

print('--- Full Directory Structure of /content/UrduSER ---')
list_all_files('/content/UrduSER/')

--- Full Directory Structure of /content/UrduSER ---
/
UrduSER A Dataset for Urdu Speech Emotion Recognition/
    UrduSER_Description.xlsx
    Sad/
        2_0_7_44.wav
        1_0_7_11.wav
        3_0_7_05.wav
        4_0_7_44.wav
        6_1_7_28.wav
        8_1_7_22.wav
        2_0_7_01.wav
        6_1_7_34.wav
        5_0_7_38.wav
        5_0_7_08.wav
        3_0_7_26.wav
        10_1_7_01.wav
        10_1_7_15.wav
        8_1_7_45.wav
        7_1_7_23.wav
        8_1_7_38.wav
        4_0_7_20.wav
        7_1_7_04.wav
        3_0_7_23.wav
        9_1_7_03.wav
        9_1_7_43.wav
        7_1_7_05.wav
        9_1_7_31.wav
        5_0_7_30.wav
        8_1_7_05.wav
        1_0_7_27.wav
        7_1_7_09.wav
        2_0_7_35.wav
        8_1_7_21.wav
        2_0_7_08.wav
        9_1_7_05.wav
        3_0_7_49.wav
        8_1_7_14.wav
        1_0_7_20.wav
        7_1_7_03.wav
        2_0_7_16.wav
        9_1_7_38.wav
        7_1_7_17.wav
        6_1_7_09.wav
        2_0_7_48.wav
        7_

In [7]:
# Re-running evaluation with correct paths and metadata processing
base_data_path = "/content/UrduSER/UrduSER A Dataset for Urdu Speech Emotion Recognition/"
metadata_path = os.path.join(base_data_path, "UrduSER_Description.xlsx")

try:
    # Reading Excel (header is usually on row 1 or 2 based on structure)
    df = pd.read_excel(metadata_path, header=1)
    file_col = "فائل کا نام"
    text_col = "تشریح"
    emotion_col = "جذبات"

    # Filtering out neutral if we want specific emotions (based on previous requests for 6 emotions)
    NEUTRAL_LABELS = {"neutral", "غیر جانبدار", ""}
    all_emotions = sorted([e for e in df[emotion_col].unique() if str(e).strip().lower() not in NEUTRAL_LABELS])

    urduser_samples = []
    for emotion in all_emotions:
        emotion_df = df[df[emotion_col] == emotion].head(NUM_SAMPLES_PER_EMOTION)
        for _, row in emotion_df.iterrows():
            file_name = str(row[file_col]).strip()
            # Search for the file in subdirectories
            audio_paths = glob.glob(f"{base_data_path}/**/{file_name}.wav", recursive=True)
            if audio_paths:
                ref_array, ref_sr = sf.read(audio_paths[0])
                if ref_array.ndim > 1: ref_array = np.mean(ref_array, axis=1)
                urduser_samples.append({
                    "audio": {"array": ref_array.astype(np.float32), "sampling_rate": int(ref_sr)},
                    "transcription": str(row[text_col]),
                    "emotion": str(emotion)
                })

    print(f"Loaded {len(urduser_samples)} samples. Starting evaluation...")
    results_urduser = run_evaluation("urduser", urduser_samples, text_key="transcription")

    df_urduser = pd.DataFrame(results_urduser)
    df_urduser.to_csv("urduser_evaluation_results_10per_emotion.csv", index=False)
    print("✅ CSV saved successfully.")
    display(df_urduser.head())
except Exception as e:
    print(f"Error: {e}")

Loaded 60 samples. Starting evaluation...


Sample 1 | Emotion: افسردہ | Similarity=0.5826
Sample 2 | Emotion: افسردہ | Similarity=0.4501
Sample 3 | Emotion: افسردہ | Similarity=0.4949
Sample 4 | Emotion: افسردہ | Similarity=0.5261
Sample 5 | Emotion: افسردہ | Similarity=0.5969
Sample 6 | Emotion: افسردہ | Similarity=0.5990
Sample 7 | Emotion: افسردہ | Similarity=0.5187
Sample 8 | Emotion: افسردہ | Similarity=0.5053
Sample 9 | Emotion: افسردہ | Similarity=0.4529
Sample 10 | Emotion: افسردہ | Similarity=0.5388
Sample 11 | Emotion: بوریت | Similarity=0.5234
Sample 12 | Emotion: بوریت | Similarity=0.5563
Sample 13 | Emotion: بوریت | Similarity=0.4398
Sample 14 | Emotion: بوریت | Similarity=0.4849
Sample 15 | Emotion: بوریت | Similarity=0.4516
Sample 16 | Emotion: بوریت | Similarity=0.5247
Sample 17 | Emotion: بوریت | Similarity=0.5568
Sample 18 | Emotion: بوریت | Similarity=0.5142
Sample 19 | Emotion: بوریت | Similarity=0.4834
Sample 20 | Emotion: بوریت | Similarity=0.5804
Sample 21 | Emotion: خوشی | Similarity=0.4373
Sample 22 | E

,sample_id,emotion,urdu_text,ref_gen_similarity
0,0,افسردہ,بہت برا ہوں,0.582558
1,1,افسردہ,میں ساری زندگی جہاں ارا سے نفرت کرتا رہا,0.450081
2,2,افسردہ,اور اس کی سزا میں نے تمہیں بھی دی,0.494872
3,3,افسردہ,دل میں تھوڑی گنجائش پیدا کر لیتا تو تین زندگیا...,0.526121
4,4,افسردہ,میں جہاں ارا اور تم ہم اکٹھے رہتے,0.596857


In [8]:
import shutil
from google.colab import files

# Re-zipping the audio folders and downloading both zip and the populated CSV
# Naming the zip specifically for the UrduSER 60-sample run
shutil.make_archive("mushra_urduser_60samples_10per_emotion", "zip", "mushra_urduser")

print("Downloading results...")
files.download("mushra_urduser_60samples_10per_emotion.zip")
files.download("urduser_evaluation_results_10per_emotion.csv")

print("✅ All files prepared for download!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files prepared for download!
